# SigLIP-2 — Fakeddit Binary Fake News Detection (No Metadata)
**2-Way Classification Only — Image + Text only, NO metadata branch**

## Model: SigLIP-2 (google/siglip-so400m-patch14-384)
- **Architecture**: Vision + Text encoders (SigLIP-2)
- **Task**: Binary classification (Real=0 vs Fake=1)
- **Dataset**: Fakeddit (5,000 balanced samples, 2-way label only)
- **Splits**: 80% Train / 10% Val / 10% Test per fold (5-fold CV)
- **Target**: >= 90% accuracy, low overfitting (gap < 5%)

## Changes from metadata version:
- ❌ Metadata branch REMOVED (no domain embeddings, no numerical features)
- ❌ 6-way classification — DISABLED
- ❌ 3-way classification — DISABLED
- ✅ 2-way classification — ONLY mode
- ✅ MixUp α=0.2, label smoothing, OneCycleLR with warmup
- ✅ Unfroze last 4 vision blocks + last 2 text blocks
- ✅ Gradient accumulation (effective batch = 32)
- ✅ Per-fold metrics table persisted live during training
- ✅ Overfitting gap displayed per fold and in final summary


In [ ]:
# ── Cell 1: Install Dependencies ─────────────────────────────────────────────
import sys, subprocess

print("Installing dependencies...\n")
packages = ["transformers>=4.40.0", "pillow", "tabulate", "scikit-learn"]
for i, pkg in enumerate(packages, 1):
    print(f"[{i}/{len(packages)}] {pkg}...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

from transformers import AutoProcessor, AutoModel
import torch
print(f"\n✓ transformers {__import__('transformers').__version__}")
print(f"✓ torch        {torch.__version__}")
print(f"✓ cuda         {torch.cuda.is_available()}")


In [ ]:
# ── Cell 2: Imports & Configuration ──────────────────────────────────────────
import os, gc, random, warnings, copy, math
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
from torch.optim.swa_utils import AveragedModel, SWALR
from PIL import Image

from transformers import AutoProcessor, AutoModel
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, f1_score,
    precision_score, recall_score,
    classification_report, confusion_matrix,
)
from tabulate import tabulate
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

# ════════════════════ CONFIGURATION ══════════════════════════════════════════
SEED             = 42
NUM_SAMPLES      = 5000
NUM_FOLDS        = 5
EPOCHS           = 15
BATCH_SIZE       = 8
ACCUM_STEPS      = 4             # effective batch = 32
LEARNING_RATE    = 3e-5
WEIGHT_DECAY     = 5e-3
DROPOUT_RATE     = 0.25
PATIENCE         = 5
LABEL_SMOOTHING  = 0.10
MIXUP_ALPHA      = 0.2
NUM_CLASSES      = 2
DEVICE           = "cuda" if torch.cuda.is_available() else "cpu"
SIGLIP_MODEL     = "google/siglip-so400m-patch14-384"
OUTPUT_DIR       = "/kaggle/working"

UNFREEZE_VIS_LAYERS = 4
UNFREEZE_TXT_LAYERS = 2

USE_SWA      = True
SWA_START_EP = max(EPOCHS - 4, 2)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

USE_AMP = (DEVICE == "cuda")

print(f"Device  : {DEVICE}")
print(f"GPU     : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"Model   : SigLIP-2 SO400M-384  (NO metadata branch)")
print(f"Task    : 2-way only  (3-way & 6-way CANCELLED)")
print(f"Config  : {EPOCHS} epochs | {NUM_FOLDS} folds | "
      f"batch {BATCH_SIZE}x{ACCUM_STEPS}={BATCH_SIZE*ACCUM_STEPS} eff.")
print(f"Unfreeze: last {UNFREEZE_VIS_LAYERS} vision + {UNFREEZE_TXT_LAYERS} text blocks")
print("=" * 70)


In [ ]:
# ── Cell 3: Fakeddit Dataset Loader ──────────────────────────────────────────
def load_fakeddit(n_samples=NUM_SAMPLES):
    """Load Fakeddit (2-way ONLY). Searches /kaggle/input for TSV/CSV."""
    print("Scanning for Fakeddit dataset...\n")
    for tsv in sorted(Path("/kaggle/input").rglob("*.tsv")):
        print(f"  Found TSV: {tsv}")
        return _load_file(tsv, n_samples)
    for csv in sorted(Path("/kaggle/input").rglob("*.csv")):
        print(f"  Found CSV: {csv}")
        return _load_file(csv, n_samples)
    raise FileNotFoundError(
        "No Fakeddit file found under /kaggle/input/\n"
        "Expected a TSV/CSV with columns: 2_way_label, clean_title/title, id"
    )


def _load_file(path, n_samples):
    sep = "\t" if path.suffix == ".tsv" else ","
    df  = pd.read_csv(path, sep=sep, low_memory=False)
    df.columns = df.columns.str.strip()
    print(f"  Columns: {list(df.columns[:10])} ...")

    text_col = next(
        (c for c in ["clean_title", "title", "text"] if c in df.columns), None
    )
    id_col = "id" if "id" in df.columns else df.columns[0]
    if "2_way_label" not in df.columns:
        raise ValueError(f"Missing '2_way_label'. Found: {list(df.columns)}")
    if not text_col:
        raise ValueError(f"No text column found in {list(df.columns)}")

    keep = [id_col, text_col, "2_way_label"]
    for c in ["image_url", "subreddit"]:
        if c in df.columns:
            keep.append(c)

    df = df[keep].dropna(subset=["2_way_label"])
    df = df.rename(columns={id_col: "id", text_col: "text",
                             "2_way_label": "label"})
    df["label"] = df["label"].astype(int)
    df = df[df["label"].isin([0, 1])]
    df["text"] = (
        df["text"].astype(str).str.strip()
        .apply(lambda x: x[:300])
        .replace("", "[No text]")
    )

    img_dir = path.parent / "public_image_set"
    if not img_dir.exists():
        for cand in path.parent.rglob("*image*"):
            if cand.is_dir():
                img_dir = cand
                break

    if img_dir.exists():
        df["image_path"] = df["id"].apply(lambda x: _find_img(x, img_dir))
        df = df[df["image_path"].notna()]
    else:
        df["image_path"] = None

    return _balance(df, n_samples)


def _find_img(img_id, img_dir):
    for ext in [".jpg", ".jpeg", ".png", ".gif", ".webp"]:
        p = img_dir / f"{img_id}{ext}"
        if p.exists():
            return str(p)
    return None


def _balance(df, n_samples):
    half   = n_samples // 2
    fake   = df[df["label"] == 1].sample(
        min(half, (df["label"] == 1).sum()), random_state=SEED)
    real   = df[df["label"] == 0].sample(
        min(half, (df["label"] == 0).sum()), random_state=SEED)
    actual = min(len(fake), len(real))
    out = (pd.concat([fake[:actual], real[:actual]])
             .sample(frac=1, random_state=SEED)
             .reset_index(drop=True))
    has_img = out["image_path"].notna().sum() if "image_path" in out.columns else 0
    print(f"\n  Loaded {len(out)} samples  (Real={actual}, Fake={actual})")
    print(f"  Images found : {has_img}")
    for i in range(min(3, len(out))):
        r = out.iloc[i]
        tag = 'FAKE' if r['label'] == 1 else 'REAL'
        print(f"  [{tag}] {str(r['text'])[:60]}...")
    return out


print("Fakeddit loader ready")


In [ ]:
# ── Cell 4: SigLIPDataset (No Metadata) ──────────────────────────────────────
class SigLIPDataset(Dataset):
    """
    Returns:
      pixel_values  (3, H, W)
      input_ids     (seq_len,)
      attention_mask(seq_len,)
      labels        ()
    No metadata features — image + text only.
    """
    PLACEHOLDER = None  # lazy init

    def __init__(self, df, processor, augment=False):
        self.df        = df.reset_index(drop=True)
        self.processor = processor
        self.augment   = augment
        if SigLIPDataset.PLACEHOLDER is None:
            SigLIPDataset.PLACEHOLDER = Image.new("RGB", (384, 384), (127, 127, 127))

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        text  = str(row["text"])
        label = int(row["label"])

        img_path = row.get("image_path") if "image_path" in row.index else None
        try:
            img = (Image.open(img_path).convert("RGB")
                   if img_path else self.PLACEHOLDER)
        except Exception:
            img = self.PLACEHOLDER

        inputs = self.processor(
            text=text, images=img,
            return_tensors="pt", padding="max_length",
            truncation=True, max_length=64,
        )
        inputs = {k: v.squeeze(0) for k, v in inputs.items()}
        inputs["labels"] = torch.tensor(label, dtype=torch.long)
        return inputs


def collate_fn(batch):
    labels = torch.stack([b["labels"] for b in batch])
    result = {"labels": labels}
    for key in batch[0]:
        if key != "labels":
            result[key] = torch.stack([b[key] for b in batch])
    return result


def mixup_batch(batch, alpha=MIXUP_ALPHA):
    """Apply MixUp to pixel_values; returns mixed, labels_a, labels_b, lam."""
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    lam = max(lam, 1 - lam)
    bs  = batch["pixel_values"].size(0)
    idx = torch.randperm(bs, device=batch["pixel_values"].device)
    mixed = {k: v.clone() for k, v in batch.items() if k != "labels"}
    mixed["pixel_values"] = (
        lam * batch["pixel_values"] + (1 - lam) * batch["pixel_values"][idx]
    )
    return mixed, batch["labels"], batch["labels"][idx], lam


print("SigLIPDataset (no metadata) + MixUp ready")


In [ ]:
# ── Cell 5: SigLIPClassifier (No Metadata) ───────────────────────────────────
class SigLIPClassifier(nn.Module):
    """
    Vision + Text dual encoder with a fusion classifier.
    No metadata branch.

    Architecture:
      img_embed (vision_dim) -> vis_proj (256) -|
      txt_embed (text_dim)   -> txt_proj (256) -+-> [512] -> LayerNorm
                                                          -> 512 -> GELU
                                                          -> 256 -> GELU
                                                          -> num_classes

    Unfreeze strategy:
      - Last UNFREEZE_VIS_LAYERS vision blocks
      - Last UNFREEZE_TXT_LAYERS text blocks
      - All projection + classifier layers (always trainable)
    """
    PROJ_DIM = 256

    def __init__(self, siglip_model,
                 num_classes=NUM_CLASSES,
                 dropout=DROPOUT_RATE):
        super().__init__()
        self.siglip     = siglip_model
        self.vision_dim = siglip_model.config.vision_config.hidden_size
        self.text_dim   = siglip_model.config.text_config.hidden_size

        # Freeze entire backbone
        for p in self.siglip.parameters():
            p.requires_grad = False

        # Unfreeze last N vision encoder blocks
        if hasattr(self.siglip.vision_model, "encoder"):
            for layer in self.siglip.vision_model.encoder.layers[-UNFREEZE_VIS_LAYERS:]:
                for p in layer.parameters():
                    p.requires_grad = True

        # Unfreeze last N text encoder blocks
        if hasattr(self.siglip.text_model, "encoder"):
            for layer in self.siglip.text_model.encoder.layers[-UNFREEZE_TXT_LAYERS:]:
                for p in layer.parameters():
                    p.requires_grad = True

        D = self.PROJ_DIM
        self.vision_proj = nn.Sequential(
            nn.Linear(self.vision_dim, D),
            nn.LayerNorm(D),
            nn.GELU(),
            nn.Dropout(0.15),
        )
        self.text_proj = nn.Sequential(
            nn.Linear(self.text_dim, D),
            nn.LayerNorm(D),
            nn.GELU(),
            nn.Dropout(0.15),
        )

        fused_dim = D + D  # 512
        self.classifier = nn.Sequential(
            nn.LayerNorm(fused_dim),
            nn.Dropout(dropout),
            nn.Linear(fused_dim, 512),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(512, 256),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(256, num_classes),
        )

        # Xavier init on all new linear layers
        for block in (self.vision_proj, self.text_proj, self.classifier):
            for m in block.modules():
                if isinstance(m, nn.Linear):
                    nn.init.xavier_uniform_(m.weight)
                    if m.bias is not None:
                        nn.init.zeros_(m.bias)

        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        total     = sum(p.numel() for p in self.parameters())
        print(f"  SigLIPClassifier (no metadata): "
              f"{trainable/1e6:.1f}M / {total/1e6:.1f}M trainable")
        print(f"  Fusion dim: {fused_dim}  (vis {D} + txt {D})")

    def forward(self, pixel_values, input_ids,
                attention_mask=None, **unused):
        out = self.siglip(
            pixel_values=pixel_values,
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True,
        )
        vis = out.image_embeds / out.image_embeds.norm(dim=-1, keepdim=True)
        txt = out.text_embeds  / out.text_embeds.norm(dim=-1, keepdim=True)
        p_vis = self.vision_proj(vis)
        p_txt = self.text_proj(txt)
        return self.classifier(torch.cat([p_vis, p_txt], dim=-1))


print("SigLIPClassifier (no metadata) defined")


In [ ]:
# ── Cell 6: Training Helpers ──────────────────────────────────────────────────

class EarlyStopping:
    """Monitor val loss; save best state, expose best train/val acc."""

    def __init__(self, patience=PATIENCE, delta=1e-4):
        self.patience       = patience
        self.delta          = delta
        self.best_score     = None
        self.counter        = 0
        self.best_state     = None
        self.best_train_acc = 0.0
        self.best_val_acc   = 0.0
        self.stop           = False

    def step(self, val_loss, val_acc, train_acc, model):
        score = -val_loss
        if self.best_score is None or score > self.best_score + self.delta:
            self.best_score     = score
            self.best_train_acc = train_acc
            self.best_val_acc   = val_acc
            self.best_state     = {k: v.cpu().clone()
                                   for k, v in model.state_dict().items()}
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True

    def restore_best(self, model):
        if self.best_state:
            model.load_state_dict(
                {k: v.to(DEVICE) for k, v in self.best_state.items()}
            )


def train_epoch(model, loader, optimizer, criterion, scaler,
                scheduler=None, use_mixup=True, accum_steps=ACCUM_STEPS):
    """Train one epoch with optional MixUp and gradient accumulation."""
    model.train()
    total_loss = correct = n = 0
    optimizer.zero_grad()
    num_batches = len(loader)

    for step, batch in enumerate(loader, 1):
        labels = batch.pop("labels").to(DEVICE)
        batch  = {k: v.to(DEVICE) for k, v in batch.items()}

        if use_mixup and MIXUP_ALPHA > 0:
            mixed, y_a, y_b, lam = mixup_batch({"labels": labels, **batch})
            mixed.pop("labels", None)
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                logits = model(**mixed)
                loss   = lam * criterion(logits, y_a) + (1 - lam) * criterion(logits, y_b)
        else:
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                logits = model(**batch)
                loss   = criterion(logits, labels)

        loss = loss / accum_steps
        scaler.scale(loss).backward()

        if step % accum_steps == 0 or step == num_batches:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            if scheduler is not None:
                scheduler.step()
            optimizer.zero_grad()

        total_loss += loss.item() * accum_steps * labels.size(0)
        correct    += (logits.argmax(1) == labels).sum().item()
        n          += labels.size(0)

        if step % 50 == 0 or step == num_batches:
            print(f"    step {step:3d}/{num_batches} | "
                  f"loss={total_loss/n:.4f} acc={correct/n:.4f}", flush=True)

    return total_loss / n, correct / n


@torch.no_grad()
def evaluate(model, loader, criterion):
    """Evaluate model; return metrics dict, predictions, true labels."""
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []

    for batch in loader:
        labels = batch.pop("labels").to(DEVICE)
        batch  = {k: v.to(DEVICE) for k, v in batch.items()}
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            logits = model(**batch)
            loss   = criterion(logits, labels)
        total_loss += loss.item() * labels.size(0)
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    y_true = np.array(all_labels)
    y_pred = np.array(all_preds)
    n      = len(y_true)
    return dict(
        loss      = total_loss / n,
        accuracy  = accuracy_score(y_true, y_pred),
        f1        = f1_score(y_true, y_pred, average="macro", zero_division=0),
        precision = precision_score(y_true, y_pred, average="macro", zero_division=0),
        recall    = recall_score(y_true, y_pred, average="macro", zero_division=0),
    ), y_pred, y_true


print("Training helpers ready")


In [ ]:
# ── Cell 7: 5-Fold Cross-Validation ──────────────────────────────────────────
# Global lists — accumulate results across folds for live display
FOLD_RESULTS = []
ALL_HISTORY  = []
ALL_CMS      = []


def _print_live_table(results):
    """Print running table after each fold completes."""
    headers = ["Fold", "Train", "Val", "Test Acc",
               "F1", "Prec", "Rec", "Ovf Gap"]
    rows = [
        [r["Fold"],
         f"{r['Train_Acc']:.4f}", f"{r['Val_Acc']:.4f}",
         f"{r['Test_Acc']:.4f}",  f"{r['F1_Macro']:.4f}",
         f"{r['Precision']:.4f}", f"{r['Recall']:.4f}",
         f"{r['Overfit_Gap']:+.4f}"]
        for r in results
    ]
    print(tabulate(rows, headers=headers, tablefmt="simple"))


def run_kfold(df):
    global FOLD_RESULTS, ALL_HISTORY, ALL_CMS
    FOLD_RESULTS.clear()
    ALL_HISTORY.clear()
    ALL_CMS.clear()

    skf     = StratifiedKFold(n_splits=NUM_FOLDS, shuffle=True, random_state=SEED)
    labels  = df["label"].values
    indices = np.arange(len(df))

    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
        free, tot = torch.cuda.mem_get_info()
        print(f"GPU available: {free/1e9:.1f} / {tot/1e9:.1f} GB\n")

    print("Loading SigLIP-2 processor and backbone...")
    processor   = AutoProcessor.from_pretrained(SIGLIP_MODEL)
    siglip_base = AutoModel.from_pretrained(SIGLIP_MODEL).cpu()
    siglip_init = {k: v.cpu().clone() for k, v in siglip_base.state_dict().items()}
    print("Backbone loaded (float32, CPU).\n")

    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

    for fold, (train_val_idx, test_idx_all) in enumerate(
            skf.split(indices, labels), 1):
        print(f"\n{'='*70}")
        print(f"  FOLD {fold} / {NUM_FOLDS}")
        print(f"{'='*70}")

        # 80 / 10 / 10 split
        # StratifiedKFold gives ~20% as test_idx_all.
        # We halve it into val (10%) and test (10%).
        rng = np.random.default_rng(SEED + fold)
        perm = rng.permutation(len(test_idx_all))
        mid       = len(perm) // 2
        val_idx   = test_idx_all[perm[:mid]]
        test_idx  = test_idx_all[perm[mid:]]
        train_idx = train_val_idx

        n_tr  = len(train_idx)
        n_va  = len(val_idx)
        n_te  = len(test_idx)
        total = n_tr + n_va + n_te
        print(f"  Train: {n_tr} ({n_tr/total*100:.0f}%) | "
              f"Val: {n_va} ({n_va/total*100:.0f}%) | "
              f"Test: {n_te} ({n_te/total*100:.0f}%)")

        # Datasets & loaders
        train_ds = SigLIPDataset(df.iloc[train_idx], processor, augment=True)
        val_ds   = SigLIPDataset(df.iloc[val_idx],   processor, augment=False)
        test_ds  = SigLIPDataset(df.iloc[test_idx],  processor, augment=False)

        pin = (DEVICE == "cuda")
        train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True,
                                  collate_fn=collate_fn,
                                  num_workers=0, pin_memory=pin)
        val_loader   = DataLoader(val_ds,   BATCH_SIZE, shuffle=False,
                                  collate_fn=collate_fn,
                                  num_workers=0, pin_memory=pin)
        test_loader  = DataLoader(test_ds,  BATCH_SIZE, shuffle=False,
                                  collate_fn=collate_fn,
                                  num_workers=0, pin_memory=pin)

        # Model setup
        print("  Initialising model...")
        siglip_base.load_state_dict(siglip_init, strict=True)
        siglip_base.to(DEVICE)

        model     = SigLIPClassifier(siglip_base).to(DEVICE)
        criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

        # Differential learning rates: backbone 10x lower than head
        backbone_params = [p for n, p in model.named_parameters()
                           if p.requires_grad and "siglip" in n]
        head_params     = [p for n, p in model.named_parameters()
                           if p.requires_grad and "siglip" not in n]
        optimizer = AdamW(
            [{"params": backbone_params, "lr": LEARNING_RATE * 0.1},
             {"params": head_params,     "lr": LEARNING_RATE}],
            weight_decay=WEIGHT_DECAY,
        )

        # OneCycleLR: warmup + cosine annealing
        total_steps = (len(train_loader) // ACCUM_STEPS + 1) * EPOCHS
        scheduler = OneCycleLR(
            optimizer,
            max_lr=[LEARNING_RATE * 0.1, LEARNING_RATE],
            total_steps=total_steps,
            pct_start=0.15,
            anneal_strategy="cos",
            div_factor=10,
            final_div_factor=1e3,
        )

        es      = EarlyStopping(patience=PATIENCE)
        swa_mdl = AveragedModel(model) if USE_SWA else None
        swa_sch = SWALR(optimizer, swa_lr=LEARNING_RATE * 0.05) if USE_SWA else None

        history = {
            "train_loss": [], "val_loss": [],
            "train_acc":  [], "val_acc":  [],
        }

        for epoch in range(1, EPOCHS + 1):
            use_mixup   = (epoch > 2)
            sched_this  = scheduler if epoch < SWA_START_EP else None
            print(f"\n  -- Epoch {epoch}/{EPOCHS}"
                  + (" [MixUp ON]" if use_mixup else "") + " --")

            tr_loss, tr_acc = train_epoch(
                model, train_loader, optimizer, criterion, scaler,
                scheduler=sched_this, use_mixup=use_mixup,
            )
            val_m, _, _ = evaluate(model, val_loader, criterion)

            history["train_loss"].append(tr_loss)
            history["val_loss"].append(val_m["loss"])
            history["train_acc"].append(tr_acc)
            history["val_acc"].append(val_m["accuracy"])

            if USE_SWA and swa_mdl is not None and epoch >= SWA_START_EP:
                swa_mdl.update_parameters(model)
                if swa_sch is not None:
                    swa_sch.step()

            es.step(val_m["loss"], val_m["accuracy"], tr_acc, model)
            marker = "  <- best" if es.counter == 0 else ""
            print(f"  E{epoch:2d} | "
                  f"TrL={tr_loss:.4f} TrA={tr_acc:.4f} | "
                  f"VaL={val_m['loss']:.4f} VaA={val_m['accuracy']:.4f}"
                  f"{marker}")

            if es.stop:
                print(f"  Early stop at epoch {epoch}")
                break

        # Restore best checkpoint
        es.restore_best(model)
        test_m, preds, true = evaluate(model, test_loader, criterion)

        # Try SWA model; keep if better
        if USE_SWA and swa_mdl is not None:
            torch.optim.swa_utils.update_bn(train_loader, swa_mdl, device=DEVICE)
            swa_mdl.eval()
            test_m_swa, preds_swa, _ = evaluate(swa_mdl, test_loader, criterion)
            if test_m_swa["accuracy"] >= test_m["accuracy"]:
                test_m = test_m_swa
                preds  = preds_swa
                print(f"  SWA used (test acc: {test_m['accuracy']:.4f})")

        cm      = confusion_matrix(true, preds)
        ALL_CMS.append(cm)

        ovf_gap = es.best_train_acc - test_m["accuracy"]
        gap_tag = "LOW - good!" if abs(ovf_gap) < 0.05 else "HIGH - overfitting!"

        print(f"\n  -- Fold {fold} Test Results --")
        print(f"  Train Acc (best ep) : {es.best_train_acc:.4f}")
        print(f"  Val Acc  (best ep)  : {es.best_val_acc:.4f}")
        print(f"  Test Accuracy       : {test_m['accuracy']:.4f}")
        print(f"  Test F1 (macro)     : {test_m['f1']:.4f}")
        print(f"  Test Precision      : {test_m['precision']:.4f}")
        print(f"  Test Recall         : {test_m['recall']:.4f}")
        print(f"  Overfitting Gap     : {ovf_gap:+.4f}  [{gap_tag}]\n")
        print(classification_report(true, preds, target_names=["Real", "Fake"]))

        result = {
            "Fold":        fold,
            "Train_Acc":   round(es.best_train_acc,   4),
            "Val_Acc":     round(es.best_val_acc,      4),
            "Test_Acc":    round(test_m["accuracy"],   4),
            "F1_Macro":    round(test_m["f1"],         4),
            "Precision":   round(test_m["precision"],  4),
            "Recall":      round(test_m["recall"],     4),
            "Overfit_Gap": round(ovf_gap,              4),
        }
        FOLD_RESULTS.append(result)
        ALL_HISTORY.append(history)

        print("\n  -- Cumulative Results Table --")
        _print_live_table(FOLD_RESULTS)

        # Free GPU memory
        siglip_base.to("cpu")
        del model, optimizer, scheduler, es, swa_mdl, swa_sch
        gc.collect()
        if DEVICE == "cuda":
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
            free, tot = torch.cuda.mem_get_info()
            print(f"  GPU after cleanup: {free/1e9:.1f}/{tot/1e9:.1f} GB\n")

    return FOLD_RESULTS, ALL_HISTORY, ALL_CMS


print("K-fold runner ready")


In [ ]:
# ── Cell 8: Load Fakeddit ────────────────────────────────────────────────────
print("Loading Fakeddit dataset (2-way, no metadata)...\n")
df_main = load_fakeddit(NUM_SAMPLES)
print(f"\nFinal dataset shape: {df_main.shape}")
df_main.head(3)


In [ ]:
# ── Cell 9: Run 5-Fold Cross-Validation ──────────────────────────────────────
# Expected runtime: ~50-80 min on Kaggle T4 GPU
print("\nStarting 5-fold cross-validation (no metadata)...\n")
fold_results, all_history, all_cms = run_kfold(df_main)


In [ ]:
# ── Cell 10: Final Results Table ─────────────────────────────────────────────
def print_final_table(fold_results):
    """Print final results table with per-fold + mean + std rows."""
    W = 82
    print("\n" + "=" * W)
    print("  SigLIP-2 (No Metadata) | Fakeddit 5000 | 2-Way | 5-Fold CV")
    print("  Target: Test Acc >= 90%  |  Overfit Gap < 5%")
    print("=" * W)

    headers = ["Fold", "Train Acc", "Val Acc", "Test Acc", "F1 Macro",
               "Precision", "Recall", "Ovf Gap", "Status"]
    rows = []
    for r in fold_results:
        acc_ok = r["Test_Acc"] >= 0.90
        gap_ok = abs(r["Overfit_Gap"]) < 0.05
        status = ("V ACC" if acc_ok else "X ACC") + " | " + ("V GAP" if gap_ok else "X GAP")
        rows.append([
            r["Fold"],
            f"{r['Train_Acc']:.4f}", f"{r['Val_Acc']:.4f}",
            f"{r['Test_Acc']:.4f}",  f"{r['F1_Macro']:.4f}",
            f"{r['Precision']:.4f}", f"{r['Recall']:.4f}",
            f"{r['Overfit_Gap']:+.4f}", status,
        ])

    accs  = [r["Test_Acc"]    for r in fold_results]
    f1s   = [r["F1_Macro"]    for r in fold_results]
    precs = [r["Precision"]   for r in fold_results]
    recs  = [r["Recall"]      for r in fold_results]
    gaps  = [r["Overfit_Gap"] for r in fold_results]
    tras  = [r["Train_Acc"]   for r in fold_results]
    vals  = [r["Val_Acc"]     for r in fold_results]

    rows.append(["------"] * len(headers))
    rows.append([
        "MEAN",
        f"{np.mean(tras):.4f}",  f"{np.mean(vals):.4f}",
        f"{np.mean(accs):.4f}",  f"{np.mean(f1s):.4f}",
        f"{np.mean(precs):.4f}", f"{np.mean(recs):.4f}",
        f"{np.mean(gaps):+.4f}", "",
    ])
    rows.append([
        "STD",
        f"{np.std(tras):.4f}",   f"{np.std(vals):.4f}",
        f"{np.std(accs):.4f}",   f"{np.std(f1s):.4f}",
        f"{np.std(precs):.4f}",  f"{np.std(recs):.4f}",
        f"{np.std(gaps):.4f}",   "",
    ])

    print(tabulate(rows, headers=headers, tablefmt="double_grid"))

    mean_acc = np.mean(accs)
    mean_gap = np.mean(gaps)
    acc_ok   = mean_acc >= 0.90
    gap_ok   = abs(mean_gap) < 0.05

    print(f"\n  {'='*28} SUMMARY {'='*28}")
    print(f"  Mean Test Accuracy  : {mean_acc:.4f}  "
          + ("[TARGET MET >= 90%]" if acc_ok
             else f"[NOT MET -- gap to 90%: {0.90 - mean_acc:.4f}]"))
    print(f"  Mean Test F1 (macro): {np.mean(f1s):.4f}")
    print(f"  Mean Precision      : {np.mean(precs):.4f}")
    print(f"  Mean Recall         : {np.mean(recs):.4f}")
    print(f"  Mean Overfit Gap    : {mean_gap:+.4f}  "
          + ("[LOW -- minimal overfitting]" if gap_ok
             else "[HIGH -- overfitting detected]"))
    print("=" * W)

    return {
        "mean_accuracy":  mean_acc,
        "mean_f1":        np.mean(f1s),
        "mean_precision": np.mean(precs),
        "mean_recall":    np.mean(recs),
        "mean_ovf_gap":   mean_gap,
    }


summary = print_final_table(fold_results)


In [ ]:
# ── Cell 11: Save Results CSV ────────────────────────────────────────────────
def save_results_csv(fold_results, output_dir=OUTPUT_DIR):
    """Save per-fold results + summary row to CSV."""
    df_r = pd.DataFrame(fold_results)
    numeric_cols = [c for c in df_r.columns if c != "Fold"]
    summary_row  = {"Fold": "MEAN+/-STD"}
    for col in numeric_cols:
        summary_row[col] = f"{df_r[col].mean():.4f}+/-{df_r[col].std():.4f}"
    df_out = pd.concat([df_r, pd.DataFrame([summary_row])], ignore_index=True)

    path = f"{output_dir}/siglip2_nometadata_results.csv"
    df_out.to_csv(path, index=False)
    print(f"Results CSV saved: {path}")
    try:
        display(df_out)
    except Exception:
        print(df_out.to_string(index=False))
    return path


save_results_csv(fold_results)


In [ ]:
# ── Cell 12: Training Curves, Confusion Matrices & Overfit Gap ───────────────
def plot_all(all_history, all_cms, output_dir=OUTPUT_DIR):
    # Training curves
    fig, axes = plt.subplots(2, NUM_FOLDS, figsize=(5 * NUM_FOLDS, 8))
    C = ["#1976D2", "#E64A19"]

    for i, h in enumerate(all_history):
        ep = range(1, len(h["train_loss"]) + 1)

        ax = axes[0, i]
        ax.plot(ep, h["train_loss"], color=C[0], lw=2, label="Train")
        ax.plot(ep, h["val_loss"],   color=C[1], lw=2, ls="--", label="Val")
        ax.set_title(f"Fold {i+1} -- Loss", fontweight="bold")
        ax.set_xlabel("Epoch")
        ax.legend()
        ax.grid(alpha=0.3)

        ax = axes[1, i]
        ax.plot(ep, h["train_acc"], color=C[0], lw=2, label="Train")
        ax.plot(ep, h["val_acc"],   color=C[1], lw=2, ls="--", label="Val")
        ax.axhline(0.90, color="green", ls=":", lw=1.5, label="90% target")
        ax.set_title(f"Fold {i+1} -- Accuracy", fontweight="bold")
        ax.set_xlabel("Epoch")
        ax.set_ylim(0.4, 1.0)
        ax.legend()
        ax.grid(alpha=0.3)

    plt.suptitle(
        "SigLIP-2 (No Metadata) -- Training Curves | 2-Way Fakeddit",
        fontsize=14, fontweight="bold", y=1.01,
    )
    plt.tight_layout()
    p1 = f"{output_dir}/training_curves.png"
    plt.savefig(p1, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {p1}")

    # Confusion matrices
    fig, axes = plt.subplots(1, NUM_FOLDS, figsize=(5 * NUM_FOLDS, 5))
    for i, cm in enumerate(all_cms):
        pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
        lbl = np.array(
            [[f"{v}\n({p:.1f}%)" for v, p in zip(rv, rp)]
             for rv, rp in zip(cm, pct)]
        )
        sns.heatmap(cm, annot=lbl, fmt="", cmap="Blues", ax=axes[i],
                    xticklabels=["Real", "Fake"],
                    yticklabels=["Real", "Fake"])
        acc_f = np.diag(cm).sum() / cm.sum()
        axes[i].set_title(f"Fold {i+1}  (acc={acc_f:.3f})", fontweight="bold")
        axes[i].set_xlabel("Predicted")
        axes[i].set_ylabel("True")

    plt.suptitle("Confusion Matrices -- 2-Way (Real vs Fake)",
                 fontsize=14, fontweight="bold", y=1.02)
    plt.tight_layout()
    p2 = f"{output_dir}/confusion_matrices.png"
    plt.savefig(p2, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {p2}")

    # Overfitting gap bar chart
    fig, ax = plt.subplots(figsize=(8, 4))
    folds  = [r["Fold"]        for r in FOLD_RESULTS]
    gaps   = [r["Overfit_Gap"] for r in FOLD_RESULTS]
    accs   = [r["Test_Acc"]    for r in FOLD_RESULTS]
    colors = ["#43A047" if abs(g) < 0.05 else "#E53935" for g in gaps]

    bars = ax.bar([f"F{f}" for f in folds], gaps,
                  color=colors, edgecolor="black", alpha=0.85)
    ax.axhline( 0.05, color="orange", ls="--", lw=1.5, label="5% threshold")
    ax.axhline(-0.05, color="orange", ls="--", lw=1.5)
    ax.axhline( 0,    color="black",  lw=0.8)
    for bar, acc in zip(bars, accs):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.002,
                f"{acc:.3f}", ha="center", fontsize=9, fontweight="bold")
    ax.set_xlabel("Fold")
    ax.set_ylabel("Overfitting Gap (Train Acc - Test Acc)")
    ax.set_title("Overfitting Gap per Fold  (green=low, red=high)",
                 fontweight="bold")
    ax.legend()
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    p3 = f"{output_dir}/overfitting_gap.png"
    plt.savefig(p3, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {p3}")


plot_all(all_history, all_cms)


In [ ]:
# ── Cell 13: Final Summary ────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("  FINAL SUMMARY -- SigLIP-2 (No Metadata) | Fakeddit | 2-Way")
print("=" * 70)
print(f"  Model       : {SIGLIP_MODEL}")
print(f"  Samples     : {NUM_SAMPLES} (balanced 50/50)")
print(f"  Folds       : {NUM_FOLDS}-fold stratified CV")
print(f"  Split       : 80% train / 10% val / 10% test (per fold)")
print(f"  Task        : 2-way ONLY  (3-way & 6-way CANCELLED)")
print(f"  Metadata    : NONE  (image + text only)")
print(f"  Unfreeze    : last {UNFREEZE_VIS_LAYERS} vision + "
      f"{UNFREEZE_TXT_LAYERS} text encoder blocks")
print(f"  Effective BS: {BATCH_SIZE} x {ACCUM_STEPS} = {BATCH_SIZE * ACCUM_STEPS}")
print(f"  Techniques  : MixUp a={MIXUP_ALPHA}, LabelSmoothing={LABEL_SMOOTHING},")
print(f"                OneCycleLR, SWA (last {EPOCHS - SWA_START_EP + 1} ep), "
      f"GradClip, WD={WEIGHT_DECAY}")
print()
print("  -- Performance --")
print(f"  Mean Test Accuracy  : {summary['mean_accuracy']:.4f}")
print(f"  Mean Test F1 Macro  : {summary['mean_f1']:.4f}")
print(f"  Mean Precision      : {summary['mean_precision']:.4f}")
print(f"  Mean Recall         : {summary['mean_recall']:.4f}")
print(f"  Mean Overfit Gap    : {summary['mean_ovf_gap']:+.4f}")
print()
acc_met = summary["mean_accuracy"] >= 0.90
gap_met = abs(summary["mean_ovf_gap"]) < 0.05
acc_msg = "MET" if acc_met else f"NOT MET -- gap {0.90 - summary['mean_accuracy']:.4f}"
gap_msg = "MET" if gap_met else "NOT MET -- overfitting detected"
print(f"  Target >= 90% Acc   : [{acc_msg}]")
print(f"  Overfit Gap < 5%    : [{gap_msg}]")
print()
print(f"  Files saved to {OUTPUT_DIR}/:")
print("    siglip2_nometadata_results.csv")
print("    training_curves.png")
print("    confusion_matrices.png")
print("    overfitting_gap.png")
print("=" * 70)
